In [ ]:
import os, sys
sys.path.append('..')
import numpy as np
import pandas as pd
# import networkx as nx
import matplotlib.pyplot as plt
from utils.evaluate import get_sensitivity
# from utils.plot import BubblePlot
outdir = os.path.join('..', 'figures')

## Load DELAY networks (ref/pred)

In [ ]:
datadir = os.path.join('..', 'DELAY', 'recalls-spillover-data')
hubs = pd.read_csv(os.path.join(datadir, 'hubsTop1000.csv'), index_col = 0)
A_pred = pd.read_csv(os.path.join(datadir, 'networkTop1000.csv'), index_col = 0)
edges_ref = pd.read_csv(os.path.join(datadir, 'refNetwork.csv'))
hubs.index = hubs.index.str.upper()
A_pred.index = A_pred.index.str.upper()
A_pred.columns = A_pred.columns.str.upper()
A_ref = pd.DataFrame(0, index = A_pred.index, columns = A_pred.columns)
for ix in edges_ref.index:
    A_ref.loc[edges_ref.Gene1.loc[ix], edges_ref.Gene2.loc[ix]] = 1
hubs['Indegree'] = A_pred.sum(0)
cols = hubs.sum(1) > 0
A_pred = A_pred.loc[cols, cols]
A_ref = A_ref.loc[cols, cols]
for ix in A_pred.index:
    A_pred.loc[ix, ix] = 0
    A_ref.loc[ix, ix] = 0

## Compute network sensitivity

In [ ]:
window = pd.Timedelta(days = 7)
datadir = os.path.join('..', 'stock-market-dataset')
fn = os.path.join(datadir, 'health_stocks_events.csv')
events = pd.read_csv(fn, index_col = 0, parse_dates = True).abs()
A_recall = get_sensitivity(window, A_pred.index, events)

In [ ]:
plt.figure(figsize = (2.5, 5))
plt.boxplot([A_recall.values[A_ref.values.astype(bool)],
             A_recall.values[A_pred.values.astype(bool)],
             A_recall.values[~A_pred.values.astype(bool)]],
             showfliers = False, patch_artist = True, widths = .5)
plt.show()

In [ ]:
A_plt = A_recall.astype(float)
A_plt.values[~A_pred.astype(bool)] = 0
A_plt.clip(lower = 0, inplace = True)

keep = ((A_plt > 0).sum(0) > 0) | ((A_plt > 0).sum(1) > 0)
A_plt = A_plt.loc[keep, keep]

G = nx.from_pandas_adjacency(A_plt, create_using = nx.DiGraph)

pos = nx.spring_layout(G)
edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])

# Extract edge list and weights
edge_list = [(u, v) for u, v, d in edges]
edge_weights = [d['weight'] for u, v, d in edges]

plt.figure(figsize = (8, 8))
# Draw graph
nx.draw(G, pos,
        with_labels=True,
        node_color='lightblue',
        edgelist=edge_list,
        edge_color=edge_weights,
        edge_cmap=plt.cm.Blues,
        node_size = 1000,
        # edge_vmin=-1,
        edge_vmax=np.quantile(A_recall.values[np.where(A_recall > 0)], .9, axis = None),
        width=2,
        arrows=True)

In [ ]:
np.quantile(sensitivity.values[np.where(sensitivity > 0)], .9, axis = None)

In [ ]:
sensitivity.plot(kind = 'box')

In [ ]:
hubs.sort_values('sensitivity', ascending = False).head(20)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (1, 4))
hubs.plot(kind = 'box', y = 'sensitivity', widths = .5, patch_artist = True, ax = ax)
ax.axhline(0, c = 'k', linestyle = '--', linewidth = .75, zorder = 0)
ax.set_xticks([])
ax.set_ylabel('Normalized Sensitivity')
plt.show()

In [ ]:
hubs.sort_values('sensitivity', ascending = False).head(20)

## Bubble plots for predicted network

In [ ]:
# To-do: separate centrality from sensitivity for presentation
# - Is centrality even that interesting in this case? What about just using forecasting/backcasting sensitivity?

In [ ]:
hubs.S_up.sort_values(ascending = False).head(20)

In [ ]:
fs1, fs2 = 7, 14
fig, ax = plt.subplots(figsize = (2.5, 2.5), subplot_kw = {'aspect' : 'equal'})
bubbles = BubblePlot(gtruth).collapse().plot(ax, 'xkcd:sky blue', fs1)
ax.set_title('Ground Truth', fontsize = fs2)
ax.axis(False)
ax.relim()
ax.autoscale_view()
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'ground-truth.svg'), bbox_inches = 'tight', dpi = 600)

In [ ]:
sp, fs1, fs2, min_deg = 2.7, 3.75, 16, 5
fig, ax = plt.subplots(figsize = (8, 8), subplot_kw = {'aspect' : 'equal'})
df = hubs#.sort_values('S_down')
df['utility'] = df.Outdegree * df.S_down
df = df.sort_values('utility')
df = df.loc[df.Outdegree >= min_deg]

c = 'xkcd:sky blue'
# cmap, vlim = plt.cm.Oranges, [0, df.S_down.max()] 
# norm = Normalize(vmin = 0, vmax = vlim[1])
# c = df.S_down.apply(lambda x: cmap(norm(x)))
# c.loc[df.S_down < 0] = 'gray'
bubbles = BubblePlot(df.Outdegree, sp).collapse().plot(ax, c, fs1)
ax.set_title('Forecasting Capacity', fontsize = fs2, y = .975)
ax.axis(False)
ax.relim()
ax.autoscale_view()
ax.annotate('  More', xy = (0, .4), xytext = (0, .875), rotation = 90, fontsize = fs2,
            ha = 'center', va = 'center', xycoords = 'axes fraction', textcoords = 'axes fraction',
            arrowprops = dict(arrowstyle='<|-', linewidth = 1.5, color = 'k', mutation_scale = 25))
ax.annotate('Less  ', xy = (0, .6), xytext = (0, .325), rotation = 90, fontsize = fs2,
            ha = 'center', va = 'center', xycoords = 'axes fraction', textcoords = 'axes fraction',
            arrowprops = dict(arrowstyle='<|-', linewidth = 1.5, color = 'k', mutation_scale = 25))
plt.tight_layout()
# plt.savefig(os.path.join(outdir, 'hubs-DELAY-network.svg'), bbox_inches = 'tight', dpi = 600)
plt.show()